First start with offensive scores.

From the clustering we have found three categories for offensive features. We used the clustering as an inspiration to find three distinct offensive categories to score the players on. This achieves less overlap/correlation between the three different scores. We have found the following categories:
* CATEGORY 1 (Playing in the Paint): posts up made/attempted, cuts made/attempted, 2-pt field goal made/attempted, screen assists (because typically job of the “big” man) 
* CATEGORY 2 (General Perimeter Offense): 3-pt FG made/attempted, assists, (points off assists), isolations made/attempted, catch and shoot made/attempted 
* CATEGORY 3 (Specialized Perimeter Offense): catch and drive made/attempted, pnr handlers made/attempted, drives made, drives with shot

We have 7, 7, and 7 features respectively. The scoring for each category is computed as follows. We fit a multivariate Gaussian on the data with the selected features and use its CDF to get a score in the range of 0 and 1. We then rank the players from worst to best and use this ranking to linearly scale the score from 0 to 100 again.


In [2]:
import sklearn
import pandas as pd
from scipy.stats import multivariate_normal

file = "train_data_yeo_all.csv"
data_full = pd.read_csv(file)

FEATURES_1 = ["posts up made", "posts up attempted", "cuts made", "cuts attempted" , "2-pt field goals made", "2-pt field goals attempted", "screen assist"]
FEATURES_2 = ["3-pt field goals made", "3-pt field goals attempted", "assists", "isolations made", "isolations attempted", 
        "catch and shoot made", "catch and shoot attempted"]
FEATURES_3 = ["catch and drive made", "catch and drive attempted", "pnr handlers made", "pnr handlers attempted", "drives made", "drives with shot"]

def fit_normal_distribution(df):
    mean = df.mean()
    cov_matrix = df.cov()
    return multivariate_normal(mean=mean, cov=cov_matrix)

distr_1 = fit_normal_distribution(data_full[FEATURES_1])
distr_2 = fit_normal_distribution(data_full[FEATURES_2])
distr_3 = fit_normal_distribution(data_full[FEATURES_3])

In [11]:
def compute_pre_scores(distr, features, player):
    return distr.cdf(player[features])

# Need scores for each player-season pair
df_player_season = data_full[["player name", "season"]].drop_duplicates().copy()
print(df_player_season.head())
# data_subset['cdf_score'] = data_subset.apply(lambda row: mvn_dist.cdf(row), axis=1)
df_player_season["cdf_score_1"] = data_full[FEATURES_1].apply(lambda row: distr_1.cdf(row), axis=1)
print("Finished 1")
df_player_season["cdf_score_2"] = data_full[FEATURES_2].apply(lambda row: distr_2.cdf(row), axis=1)
print("Finished 2")
df_player_season["cdf_score_3"] = data_full[FEATURES_3].apply(lambda row: distr_3.cdf(row), axis=1)
print("Finished 3")

# Create ranks foar each distribution, where lowest cdf_score is ranked 1
df_player_season["rank_1"] = df_player_season["cdf_score_1"].rank(ascending=True)
print("Finished 4")
df_player_season["rank_2"] = df_player_season["cdf_score_2"].rank(ascending=True)
print("Finished 5")
df_player_season["rank_3"] = df_player_season["cdf_score_3"].rank(ascending=True)
print("Finished 6")

# Compute final score where we scale the ranking to be between 0 and 100
df_player_season["score_1"] = (df_player_season["rank_1"] - 1) / (len(df_player_season) - 1) * 100
df_player_season["score_2"] = (df_player_season["rank_2"] - 1) / (len(df_player_season) - 1) * 100
df_player_season["score_3"] = (df_player_season["rank_3"] - 1) / (len(df_player_season) - 1) * 100

# Save the scores to a file
print(df_player_season.head())
df_player_season.to_csv("player_scores.csv", index=False)

          player name     season
0       Pierre Oriola  2020-2021
1  Usman Garuba Alari  2020-2021
2       Rolands Smits  2020-2021
3      Nikola Kalinic  2020-2021
4          Adam Hanga  2020-2021
Finished 1
Finished 2
Finished 3
Finished 4
Finished 5
Finished 6
          player name     season  cdf_score_1  cdf_score_2  cdf_score_3  \
0       Pierre Oriola  2020-2021     0.753711     0.002608     0.013077   
1  Usman Garuba Alari  2020-2021     0.174742     0.037289     0.041247   
2       Rolands Smits  2020-2021     0.323113     0.030058     0.010459   
3      Nikola Kalinic  2020-2021     0.404382     0.114140     0.118312   
4          Adam Hanga  2020-2021     0.049550     0.163436     0.102433   

   rank_1  rank_2  rank_3    score_1    score_2    score_3  
0   168.0  3144.0  3167.0   4.797472  90.290147  90.950876  
1  1305.0  2271.0  2653.0  37.460500  65.211146  76.185004  
2   869.0  2411.0  3235.0  24.935363  69.232979  92.904338  
3   717.0  1184.0  1916.0  20.568802  33.

We now compute the (single) defensive score which is inspired by the DPI (defensive performance indicator of the paper "Evaluation of basketball players and team performance").

DPI = BLK − BLKA + PFD − PF + STL + Deflections + LooseBallsRecovered − TOV + ScreenAssistsPTS + AST /TO

Don’t have data for: BLKA, Deflections, and LooseBallsRecovered. Hence use:

Defensive Score = BLK + PFD - PF + STL - TOV + ScreenAssistsPTS + AST / TO


In [7]:
DEF_FEATURES = ["blocks", "steals", "fouls drawn", "fouls", "turnovers", "points off screen assists", "assist to turnovers"]

df_player_season_def = data_full[["player name", "season"]].drop_duplicates().copy()
df_player_season_def["def_score"] = data_full["blocks"] + data_full["fouls drawn"] + data_full["steals"] + data_full["points off screen assists"] \
    + data_full["assists to turnovers"] - data_full["fouls"] - data_full["turnovers"]

# Create ranking for defensive score
df_player_season_def["def_rank"] = df_player_season_def["def_score"].rank(ascending=True)
# Normalize the defensive score to be in range 0-100 based on the ranking
df_player_season_def["def_score"] = (df_player_season_def["def_rank"] - 1) / (len(df_player_season_def) - 1) * 100


print(df_player_season_def.head())

# read player_scores.csv
df_player_scores = pd.read_csv("player_scores.csv")
df_player_scores["def_score"] = df_player_season_def["def_score"]
# write to player_scores.csv
df_player_scores.to_csv("player_scores.csv", index=False)

          player name     season  def_score  def_rank
0       Pierre Oriola  2020-2021  64.780236    2256.0
1  Usman Garuba Alari  2020-2021  80.149382    2791.0
2       Rolands Smits  2020-2021  29.732835    1036.0
3      Nikola Kalinic  2020-2021  83.826487    2919.0
4          Adam Hanga  2020-2021  71.128986    2477.0


As a last metric we (just) use rebounds.

In [10]:
df_player_season_reb = data_full[["player name", "season"]].drop_duplicates().copy()
df_player_season_reb["reb_rank"] = data_full["rebounds"].rank(ascending=True)
df_player_season_reb["reb_score"] = (df_player_season_reb["reb_rank"] - 1) / (len(df_player_season_reb) - 1) * 100
print(df_player_season_reb.head())

# read player_scores.csv
df_player_scores = pd.read_csv("player_scores.csv")
df_player_scores["reb_score"] = df_player_season_def["def_score"]
# write to player_scores.csv
df_player_scores.to_csv("player_scores.csv", index=False)

          player name     season  reb_rank  reb_score
0       Pierre Oriola  2020-2021    2845.0  81.700661
1  Usman Garuba Alari  2020-2021    3370.5  96.796897
2       Rolands Smits  2020-2021    2266.0  65.067509
3      Nikola Kalinic  2020-2021    2137.5  61.376041
4          Adam Hanga  2020-2021    2527.0  72.565355


In [12]:
# Create final csv file with all scores
df_final_scores = df_player_scores.drop(columns=["cdf_score_1", "cdf_score_2", "cdf_score_3", "rank_1", "rank_2", "rank_3"]).copy()
df_final_scores = df_final_scores.rename(columns={"player name": "player_name", "score_1": "off_score_1", "score_2": "off_score_2", "score_3": "off_score_3"})
print(df_final_scores.head())
df_final_scores.to_csv("player_scores.csv", index=False)

          player name     season  off_score_1  off_score_2  off_score_3  \
0       Pierre Oriola  2020-2021    95.202528     9.709853     9.049124   
1  Usman Garuba Alari  2020-2021    62.539500    34.788854    23.814996   
2       Rolands Smits  2020-2021    75.064637    30.767021     7.095662   
3      Nikola Kalinic  2020-2021    79.431198    66.015513    44.987073   
4          Adam Hanga  2020-2021    25.596093    77.621373    41.281241   

   def_score  reb_score  
0  64.780236  64.780236  
1  80.149382  80.149382  
2  29.732835  29.732835  
3  83.826487  83.826487  
4  71.128986  71.128986  
